# Exploratory Data Analysis: Influencer-Brand Mapping Dataset

## Objective
This notebook performs comprehensive exploratory data analysis (EDA) on the collected social media data from YouTube, Twitter, and Reddit. The goal is to:

1. Understand data structure and quality
2. Identify patterns and characteristics
3. Generate summary statistics for research documentation
4. Inform bot detection and feature extraction strategies

## Dataset Overview
- **YouTube**: Channels, videos, and comments from fitness influencers
- **Twitter**: User profiles and engagement data
- **Reddit**: Posts from fitness-related communities

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configure pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported successfully")

## 1. YouTube Data Analysis

### 1.1 Channel Data

In [ ]:
# Load YouTube channels data
youtube_channels = pd.read_csv('../data/raw/youtube/final_youtube_channels_clean.csv')

print(f"Dataset Shape: {youtube_channels.shape}")
print(f"\nColumn Names:\n{youtube_channels.columns.tolist()}")
print(f"\nFirst few rows:")
youtube_channels.head()

In [ ]:
# Basic statistics
print("=== YouTube Channels Summary Statistics ===")
print(f"Total Channels: {len(youtube_channels):,}")
print(f"\nSubscriber Statistics:")
print(f"  Total Subscribers: {youtube_channels['subscribers'].sum():,}")
print(f"  Average: {youtube_channels['subscribers'].mean():,.0f}")
print(f"  Median: {youtube_channels['subscribers'].median():,.0f}")
print(f"  Max: {youtube_channels['subscribers'].max():,}")
print(f"  Min: {youtube_channels['subscribers'].min():,}")

print(f"\nVideo Statistics:")
print(f"  Total Videos: {youtube_channels['total_videos'].sum():,}")
print(f"  Average per Channel: {youtube_channels['total_videos'].mean():.1f}")

print(f"\nView Statistics:")
print(f"  Total Views: {youtube_channels['total_views'].sum():,}")
print(f"  Average per Channel: {youtube_channels['total_views'].mean():,.0f}")

# Check for missing data
print(f"\nMissing Data:")
print(youtube_channels.isnull().sum())

In [ ]:
# Visualize channel size distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Subscriber distribution (log scale)
axes[0, 0].hist(np.log10(youtube_channels['subscribers'] + 1), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('log10(Subscribers)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Channel Subscriber Distribution (Log Scale)')

# Video count distribution
axes[0, 1].hist(youtube_channels['total_videos'], bins=50, edgecolor='black', color='coral')
axes[0, 1].set_xlabel('Total Videos')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Channel Video Count Distribution')

# Views distribution (log scale)
axes[1, 0].hist(np.log10(youtube_channels['total_views'] + 1), bins=50, edgecolor='black', color='green')
axes[1, 0].set_xlabel('log10(Total Views)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Channel Views Distribution (Log Scale)')

# Engagement metric: Views per subscriber
youtube_channels['views_per_subscriber'] = youtube_channels['total_views'] / (youtube_channels['subscribers'] + 1)
axes[1, 1].hist(youtube_channels['views_per_subscriber'].clip(upper=1000), bins=50, edgecolor='black', color='purple')
axes[1, 1].set_xlabel('Views per Subscriber')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Engagement: Views per Subscriber (capped at 1000)')

plt.tight_layout()
plt.savefig('../research_outputs/youtube_channels_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to research_outputs/")

### 1.2 Video Data

In [ ]:
# Load YouTube videos data
youtube_videos = pd.read_csv('../data/raw/youtube/final_youtube_videos_clean.csv')

print(f"Dataset Shape: {youtube_videos.shape}")
print(f"\nColumn Names:\n{youtube_videos.columns.tolist()}")
print(f"\nFirst few rows:")
youtube_videos.head()

In [ ]:
# Video statistics
print("=== YouTube Videos Summary Statistics ===")
print(f"Total Videos: {len(youtube_videos):,}")
print(f"Unique Channels: {youtube_videos['channel_id'].nunique():,}")

print(f"\nEngagement Statistics:")
print(f"  Total Views: {youtube_videos['views'].sum():,}")
print(f"  Total Likes: {youtube_videos['likes'].sum():,}")
print(f"  Total Comments: {youtube_videos['comments'].sum():,}")

print(f"\nAverage per Video:")
print(f"  Views: {youtube_videos['views'].mean():,.0f}")
print(f"  Likes: {youtube_videos['likes'].mean():,.0f}")
print(f"  Comments: {youtube_videos['comments'].mean():,.0f}")

# Engagement rates
youtube_videos['like_rate'] = youtube_videos['likes'] / (youtube_videos['views'] + 1)
youtube_videos['comment_rate'] = youtube_videos['comments'] / (youtube_videos['views'] + 1)

print(f"\nEngagement Rates:")
print(f"  Average Like Rate: {youtube_videos['like_rate'].mean():.4f} ({youtube_videos['like_rate'].mean()*100:.2f}%)")
print(f"  Average Comment Rate: {youtube_videos['comment_rate'].mean():.4f} ({youtube_videos['comment_rate'].mean()*100:.2f}%)")

# Check for missing data
print(f"\nMissing Data:")
print(youtube_videos.isnull().sum())

In [ ]:
# Visualize video engagement
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Views distribution
axes[0, 0].hist(np.log10(youtube_videos['views'] + 1), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('log10(Views)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Video Views Distribution (Log Scale)')

# Likes distribution
axes[0, 1].hist(np.log10(youtube_videos['likes'] + 1), bins=50, edgecolor='black', color='coral')
axes[0, 1].set_xlabel('log10(Likes)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Video Likes Distribution (Log Scale)')

# Like rate distribution
axes[1, 0].hist(youtube_videos['like_rate'].clip(upper=0.1), bins=50, edgecolor='black', color='green')
axes[1, 0].set_xlabel('Like Rate')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Video Like Rate Distribution (capped at 0.1)')

# Comment rate distribution
axes[1, 1].hist(youtube_videos['comment_rate'].clip(upper=0.01), bins=50, edgecolor='black', color='purple')
axes[1, 1].set_xlabel('Comment Rate')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Video Comment Rate Distribution (capped at 0.01)')

plt.tight_layout()
plt.savefig('../research_outputs/youtube_videos_engagement.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to research_outputs/")

### 1.3 Comment Data

In [ ]:
# Load a sample of comments data (large file)
youtube_comments = pd.read_csv('../data/raw/youtube/final_youtube_comments_clean.csv', nrows=100000)

print(f"Dataset Shape (sample): {youtube_comments.shape}")
print(f"\nColumn Names:\n{youtube_comments.columns.tolist()}")
print(f"\nFirst few rows:")
youtube_comments.head()

In [ ]:
# Comment statistics
print("=== YouTube Comments Summary Statistics (Sample) ===")
print(f"Sample Size: {len(youtube_comments):,}")
print(f"Unique Videos: {youtube_comments['video_id'].nunique():,}")

# Comment length analysis
youtube_comments['comment_length'] = youtube_comments['text'].astype(str).str.len()

print(f"\nComment Length Statistics:")
print(f"  Average: {youtube_comments['comment_length'].mean():.0f} characters")
print(f"  Median: {youtube_comments['comment_length'].median():.0f} characters")
print(f"  Min: {youtube_comments['comment_length'].min()}")
print(f"  Max: {youtube_comments['comment_length'].max()}")

print(f"\nLike Statistics:")
print(f"  Total Likes: {youtube_comments['likes'].sum():,}")
print(f"  Average per Comment: {youtube_comments['likes'].mean():.2f}")
print(f"  Comments with >0 likes: {(youtube_comments['likes'] > 0).sum():,} ({(youtube_comments['likes'] > 0).mean()*100:.1f}%)")

# Check for missing data
print(f"\nMissing Data:")
print(youtube_comments.isnull().sum())

In [ ]:
# Visualize comment characteristics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Comment length distribution
axes[0, 0].hist(youtube_comments['comment_length'].clip(upper=500), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('Comment Length (characters, capped at 500)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Comment Length Distribution')

# Comment likes distribution
axes[0, 1].hist(np.log10(youtube_comments['likes'] + 1), bins=50, edgecolor='black', color='coral')
axes[0, 1].set_xlabel('log10(Likes)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Comment Likes Distribution (Log Scale)')

# Comments per video
comments_per_video = youtube_comments.groupby('video_id').size()
axes[1, 0].hist(comments_per_video, bins=50, edgecolor='black', color='green')
axes[1, 0].set_xlabel('Comments per Video')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Comments per Video Distribution')

# Temporal analysis (if published_at exists)
if 'published_at' in youtube_comments.columns:
    youtube_comments['published_at'] = pd.to_datetime(youtube_comments['published_at'])
    youtube_comments['hour'] = youtube_comments['published_at'].dt.hour
    axes[1, 1].hist(youtube_comments['hour'], bins=24, edgecolor='black', color='purple')
    axes[1, 1].set_xlabel('Hour of Day')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Comment Posting Time Distribution')
else:
    axes[1, 1].text(0.5, 0.5, 'Timestamp data not available', ha='center', va='center')
    axes[1, 1].set_title('Comment Timing (N/A)')

plt.tight_layout()
plt.savefig('../research_outputs/youtube_comments_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to research_outputs/")

## 2. Twitter Data Analysis

In [ ]:
# Load Twitter data
twitter_matched = pd.read_csv('../data/raw/twitter/final_twitter_matched.csv')
twitter_brands = pd.read_csv('../data/raw/twitter/final_twitter_brands.csv')

print(f"Twitter Matched Users Shape: {twitter_matched.shape}")
print(f"Twitter Brands Shape: {twitter_brands.shape}")
print(f"\nMatched Users Columns:\n{twitter_matched.columns.tolist()}")
print(f"\nFirst few rows:")
twitter_matched.head()

In [ ]:
# Twitter statistics
print("=== Twitter Data Summary Statistics ===")
print(f"Total Tweets: {len(twitter_matched):,}")
print(f"Unique Influencers: {twitter_matched['influencer_name'].nunique():,}")

print(f"\nEngagement Statistics:")
print(f"  Total Likes: {twitter_matched['likes'].sum():,}")
print(f"  Total Retweets: {twitter_matched['retweets'].sum():,}")
print(f"  Total Replies: {twitter_matched['replies'].sum():,}")

print(f"\nAverage per Tweet:")
print(f"  Likes: {twitter_matched['likes'].mean():.2f}")
print(f"  Retweets: {twitter_matched['retweets'].mean():.2f}")
print(f"  Replies: {twitter_matched['replies'].mean():.2f}")

# Tweet length
twitter_matched['tweet_length'] = twitter_matched['text'].astype(str).str.len()
print(f"\nTweet Length:")
print(f"  Average: {twitter_matched['tweet_length'].mean():.0f} characters")
print(f"  Median: {twitter_matched['tweet_length'].median():.0f} characters")

# Check for missing data
print(f"\nMissing Data:")
print(twitter_matched.isnull().sum())

In [ ]:
# Visualize Twitter engagement
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Likes distribution
axes[0, 0].hist(np.log10(twitter_matched['likes'] + 1), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('log10(Likes)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Tweet Likes Distribution (Log Scale)')

# Retweets distribution
axes[0, 1].hist(np.log10(twitter_matched['retweets'] + 1), bins=50, edgecolor='black', color='coral')
axes[0, 1].set_xlabel('log10(Retweets)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Tweet Retweets Distribution (Log Scale)')

# Tweet length distribution
axes[1, 0].hist(twitter_matched['tweet_length'], bins=50, edgecolor='black', color='green')
axes[1, 0].set_xlabel('Tweet Length (characters)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Tweet Length Distribution')

# Engagement ratio
twitter_matched['engagement_ratio'] = (twitter_matched['retweets'] + 1) / (twitter_matched['likes'] + 1)
axes[1, 1].hist(twitter_matched['engagement_ratio'].clip(upper=2), bins=50, edgecolor='black', color='purple')
axes[1, 1].set_xlabel('Retweet/Like Ratio')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Tweet Engagement Ratio (capped at 2)')

plt.tight_layout()
plt.savefig('../research_outputs/twitter_engagement_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to research_outputs/")

## 3. Reddit Data Analysis

In [ ]:
# Load a sample of Reddit data (large file)
reddit_posts = pd.read_csv('../data/raw/reddit/final_reddit_posts.csv', nrows=50000)

print(f"Dataset Shape (sample): {reddit_posts.shape}")
print(f"\nColumn Names:\n{reddit_posts.columns.tolist()}")
print(f"\nFirst few rows:")
reddit_posts.head()

In [ ]:
# Reddit statistics
print("=== Reddit Posts Summary Statistics (Sample) ===")
print(f"Sample Size: {len(reddit_posts):,}")

# Engagement statistics
if 'score' in reddit_posts.columns:
    print(f"\nEngagement Statistics:")
    print(f"  Average Score: {reddit_posts['score'].mean():.2f}")
    print(f"  Median Score: {reddit_posts['score'].median():.0f}")
    print(f"  Max Score: {reddit_posts['score'].max():,}")

if 'num_comments' in reddit_posts.columns:
    print(f"\nComment Statistics:")
    print(f"  Average Comments: {reddit_posts['num_comments'].mean():.2f}")
    print(f"  Median Comments: {reddit_posts['num_comments'].median():.0f}")
    print(f"  Max Comments: {reddit_posts['num_comments'].max():,}")

# Content length
if 'selftext' in reddit_posts.columns:
    reddit_posts['content_length'] = reddit_posts['selftext'].astype(str).str.len()
    print(f"\nContent Length:")
    print(f"  Average: {reddit_posts['content_length'].mean():.0f} characters")
    print(f"  Median: {reddit_posts['content_length'].median():.0f} characters")

# Check for missing data
print(f"\nMissing Data:")
print(reddit_posts.isnull().sum())

In [ ]:
# Visualize Reddit post characteristics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Score distribution
if 'score' in reddit_posts.columns:
    axes[0, 0].hist(np.log10(reddit_posts['score'].clip(lower=1)), bins=50, edgecolor='black')
    axes[0, 0].set_xlabel('log10(Score)')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Reddit Post Score Distribution (Log Scale)')

# Comments distribution
if 'num_comments' in reddit_posts.columns:
    axes[0, 1].hist(reddit_posts['num_comments'].clip(upper=100), bins=50, edgecolor='black', color='coral')
    axes[0, 1].set_xlabel('Number of Comments (capped at 100)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Reddit Post Comments Distribution')

# Content length distribution
if 'content_length' in reddit_posts.columns:
    axes[1, 0].hist(reddit_posts['content_length'].clip(upper=1000), bins=50, edgecolor='black', color='green')
    axes[1, 0].set_xlabel('Content Length (characters, capped at 1000)')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Reddit Post Content Length Distribution')

# Subreddit distribution (top 10)
if 'subreddit' in reddit_posts.columns:
    top_subreddits = reddit_posts['subreddit'].value_counts().head(10)
    axes[1, 1].barh(range(len(top_subreddits)), top_subreddits.values, color='purple')
    axes[1, 1].set_yticks(range(len(top_subreddits)))
    axes[1, 1].set_yticklabels(top_subreddits.index)
    axes[1, 1].set_xlabel('Number of Posts')
    axes[1, 1].set_title('Top 10 Subreddits by Post Count')

plt.tight_layout()
plt.savefig('../research_outputs/reddit_posts_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to research_outputs/")

## 4. Cross-Platform Comparison

In [ ]:
# Create summary table for research paper
summary_data = {
    'Platform': ['YouTube Channels', 'YouTube Videos', 'YouTube Comments (sample)', 'Twitter Tweets', 'Reddit Posts (sample)'],
    'Records': [
        f"{len(youtube_channels):,}",
        f"{len(youtube_videos):,}",
        f"{len(youtube_comments):,}",
        f"{len(twitter_matched):,}",
        f"{len(reddit_posts):,}"
    ],
    'Key Metric 1': [
        f"Avg Subscribers: {youtube_channels['subscribers'].mean():,.0f}",
        f"Avg Views: {youtube_videos['views'].mean():,.0f}",
        f"Avg Length: {youtube_comments['comment_length'].mean():.0f} chars",
        f"Avg Likes: {twitter_matched['likes'].mean():.1f}",
        f"Avg Score: {reddit_posts['score'].mean():.1f}" if 'score' in reddit_posts.columns else "N/A"
    ],
    'Key Metric 2': [
        f"Total Videos: {youtube_channels['total_videos'].sum():,}",
        f"Avg Likes: {youtube_videos['likes'].mean():,.0f}",
        f"Avg Likes: {youtube_comments['likes'].mean():.1f}",
        f"Avg Retweets: {twitter_matched['retweets'].mean():.1f}",
        f"Avg Comments: {reddit_posts['num_comments'].mean():.1f}" if 'num_comments' in reddit_posts.columns else "N/A"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n=== Cross-Platform Dataset Summary ===")
print(summary_df.to_string(index=False))

# Save summary to file
summary_df.to_csv('../research_outputs/dataset_summary.csv', index=False)
print("\n✓ Summary saved to research_outputs/dataset_summary.csv")

## 5. Key Findings for Bot Detection

Based on this exploratory analysis, we identify the following characteristics that will inform our bot detection approach:

### YouTube
- **Subscriber-to-view ratio**: Legitimate channels show consistent patterns
- **Engagement rates**: Like and comment rates vary but follow typical distributions
- **Comment patterns**: Length, timing, and content quality can indicate bot behavior

### Twitter
- **Engagement ratios**: Retweet-to-like ratios show characteristic patterns
- **Tweet characteristics**: Length and content quality indicators
- **Temporal patterns**: Posting frequency and timing can reveal automation

### Reddit
- **Score distributions**: Legitimate posts follow specific engagement patterns
- **Content quality**: Post length and structure indicators
- **Subreddit diversity**: Bot accounts often have limited subreddit participation

## 6. Next Steps

1. **Bot Detection**: Implement heuristics and ML models based on identified patterns
2. **Feature Extraction**: Prepare CLIP/BERT embeddings using cleaned, bot-free data
3. **Documentation**: Use these visualizations and statistics in research paper methodology section

In [ ]:
print("\n" + "="*60)
print("Exploratory Data Analysis Complete!")
print("="*60)
print("\nOutputs saved to research_outputs/:")
print("  - youtube_channels_distribution.png")
print("  - youtube_videos_engagement.png")
print("  - youtube_comments_analysis.png")
print("  - twitter_engagement_analysis.png")
print("  - reddit_posts_analysis.png")
print("  - dataset_summary.csv")
print("\nReady for bot detection and feature extraction phases!")